# Benchmark: highspy vía LP (solo)

Mide tiempos del flujo productivo **write LP → highspy.readModel → run → mapeo a Pyomo**,
sin comparar con `appsi_highs` (ahorra ~1 min o más por corrida).

**Datos:** CSVs del escenario regional; años en `BENCHMARK_YEARS`.

**Hilos** (celda 1, `BENCHMARK_THREADS`):
- `0` → todos los CPUs (`os.cpu_count()`, igual que la app)
- `N > 0` → N hilos fijos (ej. `1`, `4`, `8`)
- `None` → leer `SIM_SOLVER_THREADS` del entorno

**Kernel:** `backend/.venv`

In [13]:
# Celda 1 — Setup
from __future__ import annotations

import os
import shutil
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyomo.environ as pyo
from IPython.display import display
from pyomo.core import Var

import highspy

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "backend" / "app").is_dir():
    BACKEND_ROOT = REPO_ROOT / "backend"
elif (REPO_ROOT.parent / "backend" / "app").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    BACKEND_ROOT = REPO_ROOT / "backend"
else:
    raise RuntimeError(
        "Ejecuta el notebook desde la raíz del repo o desde notebooks/"
    )

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.simulation.core.data_processing import (
    eliminar_valores_fuera_de_indices,
    get_processing_result_from_csv_dir,
    normalize_mode_of_operation_in_csv_dir,
    reorder_activity_ratio_csvs_for_dataportal,
    strip_whitespace_in_set_csvs,
)
from app.simulation.core.instance_builder import build_instance
from app.simulation.core.model_definition import create_abstract_model
from app.simulation.core.solver import _effective_solver_threads

CSV_ZIP = Path(
    "/home/jchavez/Documentos/UPME/Datos Simulacion/Regional/Caso 1/CSV.zip"
)
BENCHMARK_YEARS = {2022, 2023, 2024, 2025}
WORK_DIR = Path(tempfile.mkdtemp(prefix="osemosys_highspy_lp_"))
CSV_DIR = WORK_DIR / "csv"
LP_PATH = WORK_DIR / "model.lp"

# Opciones HiGHS (mismos defaults que solver.py; crossover "choose" en producción)
HIGHS_SOLVER_METHOD = "ipm"
HIGHS_PRESOLVE = "on"
HIGHS_PARALLEL = "on"
# "choose" = crossover solo si IPM no certifica optimalidad (recomendado HiGHS)
HIGHS_RUN_CROSSOVER = "choose"  # "on" | "off" solo para experimentos

# --- Hilos ---
BENCHMARK_THREADS: int | None = 0
_threads_config = (
    BENCHMARK_THREADS
    if BENCHMARK_THREADS is not None
    else int(os.getenv("SIM_SOLVER_THREADS", "0") or 0)
)
SOLVER_THREADS_EFFECTIVE = _effective_solver_threads(_threads_config)
if _threads_config > 0:
    SOLVER_THREADS_LABEL = str(_threads_config)
else:
    SOLVER_THREADS_LABEL = f"all({SOLVER_THREADS_EFFECTIVE})"

print("REPO_ROOT:", REPO_ROOT)
print("WORK_DIR:", WORK_DIR)
print(
    f"Hilos: config={_threads_config!r} → {SOLVER_THREADS_LABEL} "
    f"(efectivo={SOLVER_THREADS_EFFECTIVE}, cpus={os.cpu_count()})"
)

REPO_ROOT: /home/jchavez/Documentos/APPS/UPME/Osemosys_UPME
WORK_DIR: /tmp/osemosys_highspy_lp__kav6z93
Hilos: config=0 → all(16) (efectivo=16, cpus=16)


In [14]:
# Celda 2 — CSVs + instancia Pyomo

def trim_csvs_to_years(csv_dir: Path, keep_years: set[int]) -> None:
    year_csv = csv_dir / "YEAR.csv"
    if year_csv.is_file():
        df = pd.read_csv(year_csv)
        col = df.columns[0]
        df = df[df[col].astype(int).isin(keep_years)]
        df.to_csv(year_csv, index=False)
    for csv_file in csv_dir.glob("*.csv"):
        if csv_file.name == "YEAR.csv":
            continue
        df = pd.read_csv(csv_file, low_memory=False)
        if "YEAR" in df.columns:
            df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
            df = df[df["YEAR"].isin(keep_years)]
            df.to_csv(csv_file, index=False)


def unzip_csvs(zip_path: Path, dest_csv_dir: Path) -> Path:
    if not zip_path.is_file():
        raise FileNotFoundError(f"No existe: {zip_path}")
    dest_csv_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_csv_dir.parent)
    nested = dest_csv_dir.parent / "CSV"
    if nested.is_dir() and not any(dest_csv_dir.iterdir()):
        for item in nested.iterdir():
            shutil.move(str(item), str(dest_csv_dir / item.name))
        nested.rmdir()
    return dest_csv_dir


def preprocess_csv_dir(csv_dir: Path) -> None:
    reorder_activity_ratio_csvs_for_dataportal(str(csv_dir))
    normalize_mode_of_operation_in_csv_dir(str(csv_dir))
    strip_whitespace_in_set_csvs(str(csv_dir))
    eliminar_valores_fuera_de_indices(str(csv_dir))


def build_concrete_instance(csv_dir: Path):
    preprocess_csv_dir(csv_dir)
    proc = get_processing_result_from_csv_dir(str(csv_dir))
    has_storage = proc.has_storage
    has_udc = proc.has_udc
    print(f"has_storage={has_storage}, has_udc={has_udc}")
    print(
        f"REGION={len(proc.sets.get('REGION', []))}, "
        f"TECH={len(proc.sets.get('TECHNOLOGY', []))}, "
        f"YEAR={len(proc.sets.get('YEAR', []))}, "
        f"TS={len(proc.sets.get('TIMESLICE', []))}"
    )
    t0 = perf_counter()
    abstract = create_abstract_model(has_storage=has_storage, has_udc=has_udc)
    instance = build_instance(
        abstract,
        str(csv_dir),
        has_storage=has_storage,
        has_udc=has_udc,
    )
    return instance, perf_counter() - t0


csv_dir = unzip_csvs(CSV_ZIP, CSV_DIR)
trim_csvs_to_years(csv_dir, BENCHMARK_YEARS)
print("Años:", sorted(BENCHMARK_YEARS))

instance, build_seconds = build_concrete_instance(csv_dir)
print(f"Instancia construida en {build_seconds:.2f} s")

Años: [2022, 2023, 2024, 2025]
has_storage=True, has_udc=False
REGION=1, TECH=2343, YEAR=4, TS=1
           0 seconds to construct Set YEAR; 1 index total
           0 seconds to construct Set TECHNOLOGY; 1 index total
           0 seconds to construct Set TIMESLICE; 1 index total
           0 seconds to construct Set FUEL; 1 index total
           0 seconds to construct Set EMISSION; 1 index total
           0 seconds to construct Set MODE_OF_OPERATION; 1 index total
           0 seconds to construct Set REGION; 1 index total
           0 seconds to construct Set STORAGE; 1 index total
           0 seconds to construct Set SEASON; 1 index total
           0 seconds to construct Set DAYTYPE; 1 index total
           0 seconds to construct Set DAILYTIMEBRACKET; 1 index total
           0 seconds to construct Set STORAGEINTRADAY; 1 index total
           0 seconds to construct Set STORAGEINTRAYEAR; 1 index total
           0 seconds to construct Set FLEXIBLEDEMANDTYPE; 1 index total
    

In [15]:
# Celda 3 — highspy vía LP (write + read + solve + mapeo primals)


def pyomo_name_to_lp(name: str) -> str:
    if "[" in name and name.endswith("]"):
        base, rest = name.split("[", 1)
        return f"{base}({rest[:-1]})"
    return name


def lp_name_to_pyomo(name: str) -> str:
    if "(" in name and name.endswith(")"):
        base, rest = name.split("(", 1)
        return f"{base}[{rest[:-1]}]"
    return name


def highs_status_label(status: object) -> str:
    mapping = {
        getattr(highspy.HighsModelStatus, "kOptimal", None): "optimal",
        getattr(highspy.HighsModelStatus, "kInfeasible", None): "infeasible",
        getattr(highspy.HighsModelStatus, "kUnbounded", None): "unbounded",
    }
    for hs, label in mapping.items():
        if hs is not None and status == hs:
            return label
    return str(status)


def apply_highspy_solution(instance, h: highspy.Highs) -> float:
    solution = h.getSolution()
    lp = h.getLp()
    col_names = list(getattr(lp, "col_names_", []) or [])
    col_values = list(getattr(solution, "col_value", []) or [])
    col_map: dict[str, float] = {}
    for idx, name in enumerate(col_names):
        if idx < len(col_values):
            val = float(col_values[idx])
            col_map[name] = val
            col_map[lp_name_to_pyomo(name)] = val
    for var in instance.component_data_objects(Var, active=True):
        pyomo_name = var.name
        lp_name = pyomo_name_to_lp(pyomo_name)
        val = col_map.get(pyomo_name) or col_map.get(lp_name)
        if val is not None:
            var.set_value(val, skip_validation=True)
    try:
        return float(h.getInfo().objective_function_value)
    except Exception:
        return float(pyo.value(instance.OBJ))


def run_highspy_via_lp(
    instance,
    lp_path: Path,
    *,
    threads: int,
    solver_method: str = HIGHS_SOLVER_METHOD,
    presolve: str = HIGHS_PRESOLVE,
    parallel: str = HIGHS_PARALLEL,
    run_crossover: str = HIGHS_RUN_CROSSOVER,
) -> dict:
    lp_path = Path(lp_path)
    lp_path.parent.mkdir(parents=True, exist_ok=True)

    t_write = perf_counter()
    instance.write(
        filename=str(lp_path),
        io_options={"symbolic_solver_labels": True},
    )
    write_lp_seconds = perf_counter() - t_write
    lp_size_mb = lp_path.stat().st_size / (1024 * 1024)

    h = highspy.Highs()
    h.setOptionValue("log_to_console", False)
    h.setOptionValue("output_flag", False)
    h.setOptionValue("solver", solver_method)
    h.setOptionValue("presolve", presolve)
    h.setOptionValue("parallel", parallel)
    h.setOptionValue("run_crossover", run_crossover)
    h.setOptionValue("threads", threads)

    t_read = perf_counter()
    h.readModel(str(lp_path))
    read_model_seconds = perf_counter() - t_read

    t_run = perf_counter()
    h.run()
    run_seconds = perf_counter() - t_run

    status = highs_status_label(h.getModelStatus())
    obj = 0.0
    map_seconds = 0.0
    if "optimal" in status.lower():
        t_map = perf_counter()
        obj = apply_highspy_solution(instance, h)
        map_seconds = perf_counter() - t_map

    return {
        "status": status,
        "objective": obj,
        "build_seconds": build_seconds,
        "write_lp_seconds": write_lp_seconds,
        "read_model_seconds": read_model_seconds,
        "run_seconds": run_seconds,
        "map_solution_seconds": map_seconds,
        "total_seconds": (
            build_seconds
            + write_lp_seconds
            + read_model_seconds
            + run_seconds
            + map_seconds
        ),
        "solve_pipeline_seconds": (
            write_lp_seconds + read_model_seconds + run_seconds + map_seconds
        ),
        "lp_size_mb": lp_size_mb,
        "solver_method": solver_method,
        "presolve": presolve,
        "parallel": parallel,
        "run_crossover": run_crossover,
        "threads_config": SOLVER_THREADS_LABEL,
        "threads_effective": threads,
    }


result = run_highspy_via_lp(
    instance,
    LP_PATH,
    threads=SOLVER_THREADS_EFFECTIVE,
)
print("highspy via LP:", result)
print(f"LP: {LP_PATH} ({result['lp_size_mb']:.2f} MB)")

highspy via LP: {'status': 'HighsModelStatus.kUnknown', 'objective': 0.0, 'build_seconds': 4.16153811500044, 'write_lp_seconds': 7.924102422999567, 'read_model_seconds': 2.4430919959995663, 'run_seconds': 3.749796789999891, 'map_solution_seconds': 0.0, 'total_seconds': 18.278529323999464, 'solve_pipeline_seconds': 14.116991208999025, 'lp_size_mb': 110.74879837036133, 'solver_method': 'ipm', 'presolve': 'on', 'parallel': 'on', 'run_crossover': 'off', 'threads_config': 'all(16)', 'threads_effective': 16}
LP: /tmp/osemosys_highspy_lp__kav6z93/model.lp (110.75 MB)


In [16]:
# Celda 4 — Resumen de tiempos

df = pd.DataFrame(
    [
        {"fase": "build_instance", "segundos": result["build_seconds"]},
        {"fase": "write_lp", "segundos": result["write_lp_seconds"]},
        {"fase": "read_model", "segundos": result["read_model_seconds"]},
        {"fase": "highs_run", "segundos": result["run_seconds"]},
        {"fase": "map_solution", "segundos": result["map_solution_seconds"]},
        {"fase": "solve_pipeline (sin build)", "segundos": result["solve_pipeline_seconds"]},
        {"fase": "TOTAL (build + solve)", "segundos": result["total_seconds"]},
    ]
)
display(df)
print(
    f"status={result['status']}, objective={result['objective']:.4f}, "
    f"hilos={result['threads_config']}"
)

,fase,segundos
0,build_instance,4.161538
1,write_lp,7.924102
2,read_model,2.443092
3,highs_run,3.749797
4,map_solution,0.000000
5,solve_pipeline (sin build),14.116991
6,TOTAL (build + solve),18.278529


status=HighsModelStatus.kUnknown, objective=0.0000, hilos=all(16)


In [17]:
# Celda 5 (opcional) — Barrido de hilos reutilizando el .lp ya escrito
# Solo readModel + run (sin rewrite LP ni rebuild). Puede tardar varios minutos.

THREAD_SWEEP = False  # True para comparar 1 vs 4 vs all(cpus)

if not THREAD_SWEEP:
    print("THREAD_SWEEP=False — omitido.")
elif not LP_PATH.is_file():
    raise FileNotFoundError("Ejecuta la celda 3 primero.")
else:
    cpu_n = os.cpu_count() or 1
    sweep_configs = [
        (1, "1"),
        (4, "4"),
        (_effective_solver_threads(0), f"all({cpu_n})"),
    ]
    sweep_rows = []
    for threads_n, label in sweep_configs:
        h = highspy.Highs()
        h.setOptionValue("log_to_console", False)
        h.setOptionValue("output_flag", False)
        h.setOptionValue("solver", HIGHS_SOLVER_METHOD)
        h.setOptionValue("presolve", HIGHS_PRESOLVE)
        h.setOptionValue("parallel", HIGHS_PARALLEL)
        h.setOptionValue("run_crossover", HIGHS_RUN_CROSSOVER)
        h.setOptionValue("threads", threads_n)
        t0 = perf_counter()
        h.readModel(str(LP_PATH))
        read_s = perf_counter() - t0
        t1 = perf_counter()
        h.run()
        run_s = perf_counter() - t1
        sweep_rows.append({
            "hilos": label,
            "threads_effective": threads_n,
            "read_model_seconds": read_s,
            "run_seconds": run_s,
            "total_seconds": read_s + run_s,
            "status": highs_status_label(h.getModelStatus()),
        })
    display(pd.DataFrame(sweep_rows))

THREAD_SWEEP=False — omitido.


In [18]:
# Opcional: limpiar WORK_DIR
# shutil.rmtree(WORK_DIR, ignore_errors=True)